# Practice Test 1: Mechanistic Interpretability Fundamentals

**Set a 120-minute timer before you start.**

Rules:
- Implement everything from scratch (no copying from practice notebooks)
- NumPy/Python docs are allowed
- No AI tools, no pasting external code


In [1]:
import numpy as np

def check(name, got, expected=None, cond=None, rtol=1e-4, atol=1e-6):
    try:
        if cond is not None:
            passed = bool(cond(got))
        elif expected is not None:
            passed = np.allclose(np.array(got, dtype=float),
                                 np.array(expected, dtype=float), rtol=rtol, atol=atol)
        else:
            passed = True
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed and expected is not None:
            g, e = np.array(got), np.array(expected)
            print(f"         got shape {g.shape}, expected shape {e.shape}")
            if g.shape == e.shape and g.size <= 16:
                print(f"         got:      {g.ravel()}")
                print(f"         expected: {e.ravel()}")
        return passed
    except Exception as exc:
        print(f"  [ERROR] {name}: {type(exc).__name__}: {exc}")
        return False

print("Helper loaded. check() is available.")


Helper loaded. check() is available.


## Q1: Softmax Family

In [2]:
def log_softmax(z, axis=-1):
    """
    Goal: Compute the log-softmax of a tensor in a numerically stable way to prevent overflow/underflow.
    Do NOT compute the softmax first and then take the log, as that defeats the numerical stability.
    
    Args:
        z: np.ndarray of arbitrary shape. The unnormalized logits.
        axis: int. The axis along which to compute the log-softmax.
        
    Returns:
        np.ndarray of the exact same shape as z, containing the log-probabilities.
    """
    zmax = z.max(axis=axis, keepdims=True)
    z -= zmax
    return z - np.log( np.exp(z).sum(axis=axis, keepdims=True) )


In [3]:
print("="*50 + "\nQ1 TESTS\n" + "="*50)
z = np.array([1., 2., 3.])
expected_ls = np.array([-2.4076, -1.4076, -0.4076])
check("log_softmax basic", log_softmax(z), expected_ls, rtol=1e-3)
check("log_softmax stable on large z",
      cond=lambda x: np.all(np.isfinite(x)), got=log_softmax(np.array([1000., 1001., 1002.])))


Q1 TESTS
  [PASS] log_softmax basic
  [PASS] log_softmax stable on large z


True

## Q2: Optimal einsum Contraction

In [4]:
def optimal_path_contraction(W_E, W_V, W_O, W_U):
    """
    Goal: Compute the end-to-end direct path transition matrix. If contracted poorly 
    (e.g., W_E @ W_U first), it will create a massive (V, V) intermediate tensor and OOM.
    Use a single `np.einsum` call with the optimize flag to find the most efficient path.
    
    Args:
        W_E: (V, d) Token embedding matrix.
        W_V: (d, dh) Value projection matrix.
        W_O: (dh, d) Output projection matrix.
        W_U: (d, V) Unembedding matrix.
        
    Returns:
        (V, V) transition matrix mapping input vocab probabilities to output vocab probabilities.
    """
    return np.einsum("vm,ma,an,nw->vw", W_E, W_V, W_O, W_U, optimize="optimal")
    


In [5]:
print("="*50 + "\nQ2 TESTS\n" + "="*50)
V_voc, d, dh = 10, 8, 4
np.random.seed(42)
W_E = np.random.randn(V_voc, d); W_V = np.random.randn(d, dh)
W_O = np.random.randn(dh, d); W_U = np.random.randn(d, V_voc)
out = optimal_path_contraction(W_E, W_V, W_O, W_U)
expected_out = W_E @ W_V @ W_O @ W_U
check("shape (V, V)", cond=lambda x: x.shape == (V_voc, V_voc), got=out)
check("matches naive matmul", out, expected_out)


Q2 TESTS
  [PASS] shape (V, V)
  [PASS] matches naive matmul


True

## Q3: Skip-Trigram Attribution

In [37]:
def get_skip_trigram_scores(W_E, W_QK, W_OV, W_U):
    """
    Goal: In a 1-layer attention-only model, heads act as skip-trigrams (A... B -> C). 
    Calculate the raw, unnormalized score for predicting token C given destination token B 
    attending to source token A.
    
    Args:
        W_E: (V, d) Token embedding matrix.
        W_QK: (d, d) The combined QK circuit matrix (W_Q @ W_K.T).
        W_OV: (d, d) The combined OV circuit matrix (W_V @ W_O).
        W_U: (d, V) Unembedding matrix.
        
    Returns:
        (V, V, V) tensor where out[a, b, c] is the attribution score.
    """
    ab_score = np.einsum("bm,mn,an->ab",W_E, W_QK, W_E, optimize="optimal") # V by V unnormalized
    out_logs = np.einsum("ns,sc->nc",  W_OV, W_U, optimize="optimal" ) # V by V
    ans=np.einsum( "ab,an,nc->abc" , ab_score, W_E, out_logs, optimize="optimal")
    
    print(ans.shape)
    return  ans
    
#     raise NotImplementedError()


In [38]:
print("="*50 + "\nQ3 TESTS\n" + "="*50)
V_voc, d = 5, 4
np.random.seed(1)
W_E = np.random.randn(V_voc, d); W_QK = np.random.randn(d, d)
W_OV = np.random.randn(d, d); W_U = np.random.randn(d, V_voc)
out = get_skip_trigram_scores(W_E, W_QK, W_OV, W_U)
check("shape (V, V, V)", cond=lambda x: x.shape == (V_voc, V_voc, V_voc), got=out)
a, b, c = 1, 2, 3
expected_val = (W_E[b] @ W_QK @ W_E[a].T) * (W_E[a] @ W_OV @ W_U[:, c])
check("matches manual derivation for T[a,b,c]", out[a, b, c], expected_val)


Q3 TESTS
(5, 5, 5)
  [PASS] shape (V, V, V)
  [PASS] matches manual derivation for T[a,b,c]


True

## Q4: Residual Stream Analysis (3 Parts)

In [20]:
# Part 1: Direct Path
def direct_path_logits(tokens, W_E, W_U):
    """
    Goal: Calculate the 0-layer direct path logits (bigram statistics) directly 
    from the residual stream, skipping all attention and MLP blocks.
    
    Args:
        tokens: (T,) 1D array of token integers.
        W_E: (V, d) Embedding matrix.
        W_U: (d, V) Unembedding matrix.
        
    Returns:
        (T, V) array of logits.
    """
    print( f"W_E shape = {W_E.shape}")
    print(f"tokens length = {len(tokens)}")
    return W_E[tokens, :] @ W_U

# Part 2: Head-to-Logit Attribution
def head_logit_attribution(attention_pattern, X, W_V, W_O, W_U):
    """
    Goal: Calculate the exact contribution of a single attention head to the 
    final vocabulary logits at each sequence position.
    
    Args:
        attention_pattern: (T, T) The post-softmax attention weights for this head.
        X: (T, d) The residual stream input to the head.
        W_V: (d, dh) Value projection.
        W_O: (dh, d) Output projection.
        W_U: (d, V) Unembedding matrix.
        
    Returns:
        (T, V) array representing the logit shifts caused by this head.
    """
    A = attention_pattern
    return np.einsum( "ij,jm,ma,an,nw->iw" , A, X, W_V, W_O, W_U, optimize=True)
    
    

# Part 3: Q-Composition
def q_composition_weights(W_OV_L1, W_Q_L2):
    """
    Goal: Compute the virtual weights describing how the Layer 2 Query circuit 
    reads from the subspace written to by the Layer 1 Output-Value circuit.
    
    Args:
        W_OV_L1: (d, d) The combined OV circuit for a head in Layer 1.
        W_Q_L2: (d, dh) The Query projection for a head in Layer 2.
        
    Returns:
        (d, dh) array of virtual query weights.
    """
    return np.einsum("mn,na->ma", W_OV_L1, W_Q_L2)
    


In [21]:
print("Q4 Part 1-3 Tests")
tokens = np.array([0, 3, 1])
out1 = direct_path_logits(tokens, W_E, W_U)
check("Part 1: shape (T, V)", cond=lambda x: x.shape == (3, V_voc), got=out1)

T_seq = 3
attn = np.random.randn(T_seq, T_seq)
X = np.random.randn(T_seq, d)
W_V_h = np.random.randn(d, dh); W_O_h = np.random.randn(dh, d)
out2 = head_logit_attribution(attn, X, W_V_h, W_O_h, W_U)
check("Part 2: shape (T, V)", cond=lambda o: o.shape == (T_seq, V_voc), got=out2)

out3 = q_composition_weights(W_OV, W_V_h) # using W_V_h shape as dummy for W_Q
check("Part 3: shape (d, dh)", cond=lambda x: x.shape == (d, dh), got=out3)


Q4 Part 1-3 Tests
W_E shape = (5, 4)
tokens length = 3
  [PASS] Part 1: shape (T, V)
  [PASS] Part 2: shape (T, V)
  [PASS] Part 3: shape (d, dh)


True